# Re-run a Spectre transient sim from its previous run

Reconstructed from a prior run's command line (in its `psf/spectre.out`). Set **`RUN_DIR`** below
to that run's **netlist directory** (the folder containing `input.scs`).
**Run where Cadence Spectre + the PDK are installed.**

The rerun is **non-destructive**: it logs to `spectre_rerun.out` and a separate `psf_rerun/`
raw dir, so the original `spectre.out` / `psf/` are left untouched.

Run the cells top-to-bottom (**Shift+Enter**).

## 1. Config — paths & command (from the original log's "Command line:")

In [ ]:
import os, socket, subprocess, shlex, re, shutil, glob

# ---- USER INPUT: netlist dir of a previous Spectre run (the folder with input.scs) ----
#   example: RUN_DIR = "/path/to/run/.../netlist"
RUN_DIR = ""        # <-- SET ME
AHDLLIB = None      # optional: AHDL/Verilog-A include dir if the run used it, else None

# Spectre is often NOT on $PATH. Resolve: SPECTRE_BIN env > the binary recorded in this run's
# psf/spectre.out > $PATH > common Cadence install dirs. License matched to that install.
def find_spectre(run_dir):
    if os.environ.get("SPECTRE_BIN"):
        return os.environ["SPECTRE_BIN"]
    log = os.path.join(run_dir, "..", "psf", "spectre.out") if run_dir else ""
    if log and os.path.exists(log):
        m = re.search(r"(/[^\s'\"]+/bin/spectre)\b", open(log).read())
        if m and os.path.exists(m.group(1)):
            return m.group(1)
    if shutil.which("spectre"):
        return shutil.which("spectre")
    hits = sorted({x for p in ["/usr/local/packages/*/SPECTRE*/tools.lnx86/bin/spectre",
                                "/opt/cadence*/*/SPECTRE*/tools*/bin/spectre"] for x in glob.glob(p)})
    return hits[-1] if hits else "spectre"

SPECTRE = find_spectre(RUN_DIR)
if not (os.environ.get("CDS_LIC_FILE") or os.environ.get("LM_LICENSE_FILE")):
    if os.environ.get("CDS_LIC_OVERRIDE"):
        os.environ["CDS_LIC_FILE"] = os.environ["CDS_LIC_OVERRIDE"]
    else:                                                  # walk up to a sibling license.dat
        d = os.path.dirname(os.path.abspath(SPECTRE))
        for _ in range(6):
            if os.path.exists(os.path.join(d, "license.dat")):
                os.environ["CDS_LIC_FILE"] = os.path.join(d, "license.dat"); break
            d = os.path.dirname(d)

NETLIST   = RUN_DIR
RERUN_LOG = "../psf/spectre_rerun.out"   # relative to NETLIST; redirected so originals are safe
RERUN_RAW = "../psf_rerun"

cmd = [SPECTRE, "-64", "input.scs", "+escchars", "+log", RERUN_LOG,
       "-format", "psfxl", "-raw", RERUN_RAW,
       "+aps", "+lqtimeout", "900", "-maxw", "5", "-maxn", "5", "+logstatus"]
if AHDLLIB:
    cmd += ["-env", "ade", "-ahdllibdir", AHDLLIB]

found = bool(shutil.which(SPECTRE) or os.path.exists(SPECTRE))
print("host        :", socket.gethostname())
print("netlist dir :", NETLIST or "(SET RUN_DIR!)")
print("spectre     :", SPECTRE, "(found)" if found else "(NOT FOUND — set SPECTRE_BIN)")
print("license     :", os.environ.get("CDS_LIC_FILE", "(ambient env)"))
print("input.scs   :", "found" if NETLIST and os.path.exists(f"{NETLIST}/input.scs")
                        else "MISSING — set RUN_DIR")
print("\ncommand:\n ", " ".join(shlex.quote(c) for c in cmd))

## 2. Run it (live output streams below)

In [ ]:
assert NETLIST and os.path.exists(os.path.join(NETLIST, "input.scs")), \
    "Set RUN_DIR (cell above) to a run's netlist dir that contains input.scs."
assert shutil.which(SPECTRE) or os.path.exists(SPECTRE), \
    "spectre not found — set SPECTRE_BIN or put spectre on PATH."

# cwd MUST be the netlist dir — input.scs uses relative includes.
proc = subprocess.run(cmd, cwd=NETLIST, text=True,
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
out = proc.stdout
print(out[-4000:])   # tail of stdout
print(f"\n[exit code: {proc.returncode}]")

## 3. Parse & summarize the result

In [ ]:
# Read the freshly written rerun log and pull out the headline numbers.
log_path = os.path.join(NETLIST, RERUN_LOG)
text = open(log_path).read() if os.path.exists(log_path) else out

def grab(pat):
    m = re.search(pat, text)
    return m.group(1).strip() if m else "(not found)"

print("=== RERUN SUMMARY ===")
print("completion :", grab(r"(spectre completes with .*?)\."))
print("tran steps :", grab(r"Number of accepted tran steps =\s+(\d+)"))
print("max V      :", grab(r"V:\s*(V\([^)]*\) = .*?V)"))
print("max I      :", grab(r"I:\s*(I\([^)]*\) = .*?A)"))
print("wall clock :", grab(r"elapsed time \(wall clock\):\s*(.*?)\."))
print("peak mem   :", grab(r"Peak memory used = (\d+ Mbytes)"))

errs = re.search(r"completes with (\d+) errors", text)
print("\nPASS" if errs and errs.group(1) == "0" else "\nCHECK — errors present")

## 4. (Optional) Plot a transient waveform from the rerun rawfile
Needs a PSF reader. If you have Cadence's Python `psf`/`libpsf` available you can load `psf_rerun/tran.tran.tran`. Otherwise skip — the summary above already confirms the run.

In [ ]:
raw = os.path.join(NETLIST, RERUN_RAW)
print("raw outputs:")
for f in sorted(os.listdir(raw)) if os.path.isdir(raw) else []:
    print("  ", f)